# Buzzlytics — Stage-2 Varroa Classifier (YOLOv8-cls: healthy / varroa)

Trains a small **YOLOv8n-cls** image classifier on VarroaDataset single-bee crops.
At inference, the pipeline crops each bee the detector found and runs this model to
decide `healthy` vs `varroa` — flipping a box to `varroa_bee` when infected.

## Prep
Run `build_dataset_colab.ipynb` once to produce **`varroa_cls.zip`** in your Drive,
then share-link it and paste below.

## Run
1. Runtime → GPU. 2. Paste `DATASET_URL`. 3. Run all. 4. Download `varroa_cls.pt`
→ place at `cv_pipeline/weights/varroa_cls.pt` in your repo.

In [ ]:
# === 0. Configuration ===
# Drive share link to varroa_cls.zip (Anyone-with-the-link).
DATASET_URL = "PASTE_GDRIVE_SHARE_LINK_HERE"

EPOCHS = 30
IMGSZ  = 128     # crops are small single bees; 128 is plenty
BATCH  = 64
MODEL  = "yolov8n-cls.pt"

assert DATASET_URL != "PASTE_GDRIVE_SHARE_LINK_HERE", "Paste your Google Drive share link first"

In [ ]:
# === 1. Confirm GPU ===
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (switch Runtime -> GPU!)")

In [ ]:
# === 2. Install dependencies ===
!pip -q install "ultralytics==8.4.71" gdown

In [ ]:
# === 3. Download varroa_cls.zip + extract + locate the ImageFolder root ===
import gdown, zipfile
from pathlib import Path

gdown.download(url=DATASET_URL, output="/content/varroa_cls.zip", quiet=False, fuzzy=True)
with zipfile.ZipFile("/content/varroa_cls.zip") as z:
    z.extractall("/content/varroa_cls_data")

# Root = the dir that directly contains train/ (with class subfolders).
CLS_DIR = None
for p in Path("/content/varroa_cls_data").rglob("train/varroa"):
    CLS_DIR = str(p.parent.parent)
    break
assert CLS_DIR, "No train/varroa folder found. Re-zip varroa_cls with train/ val/ each having healthy/ + varroa/."
print("CLS_DIR:", CLS_DIR)
for split in ["train", "val", "test"]:
    for lab in ["healthy", "varroa"]:
        d = Path(CLS_DIR) / split / lab
        print(f"  {split}/{lab}: {len(list(d.glob('*'))) if d.is_dir() else 0}")

In [ ]:
# === 4. Train the classifier ===
from ultralytics import YOLO
model = YOLO(MODEL)
results = model.train(
    data=CLS_DIR,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    name="varroa_cls",
    patience=10,
    seed=42,
)

In [ ]:
# === 5. Validate (top-1 accuracy + per-class) ===
best = "runs/classify/varroa_cls/weights/best.pt"
m = YOLO(best).val(data=CLS_DIR, split="test")
print("top-1 accuracy:", round(float(m.top1), 4))
print("top-5 accuracy:", round(float(m.top5), 4))

In [ ]:
# === 6. Download as varroa_cls.pt (place at cv_pipeline/weights/varroa_cls.pt) ===
import shutil
from google.colab import files
shutil.copy("runs/classify/varroa_cls/weights/best.pt", "/content/varroa_cls.pt")
files.download("/content/varroa_cls.pt")